In [4]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
!pip install openai anthropic chromadb pdfplumber langsmith --quiet

print("✅ Dependencies installed")

✅ Dependencies installed


In [5]:
# ============================================================
# CELL 2 — Load secrets + mount Google Drive
# ============================================================
from google.colab import userdata, drive
import os

# Load API keys
os.environ["OPENAI_API_KEY"]       = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"]    = userdata.get("ANTHROPIC_API_KEY")
os.environ["LANGSMITH_API_KEY"]    = userdata.get("LANGSMITH_API_KEY")
os.environ["LANGCHAIN_PROJECT"]    = userdata.get("LANGSMITH_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

# Mount Google Drive — this will prompt you to authenticate
drive.mount("/content/drive")

# Build path to your PDF folder
FOLDER_NAME  = userdata.get("GOOGLE_DRIVE_FOLDER")  # "salesforce-rag-docs"
DOCS_PATH    = f"/content/drive/MyDrive/{FOLDER_NAME}"

print(f"✅ Secrets loaded")
print(f"✅ Drive mounted")
print(f"📁 Looking for PDFs in: {DOCS_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Secrets loaded
✅ Drive mounted
📁 Looking for PDFs in: /content/drive/MyDrive/salesforce-rag-docs


In [6]:
# ============================================================
# CELL 3 — Verify PDFs are accessible
# ============================================================
import os

pdf_files = [f for f in os.listdir(DOCS_PATH) if f.endswith(".pdf")]
pdf_files.sort()

print(f"Found {len(pdf_files)} PDFs:\n")
for f in pdf_files:
    size_kb = os.path.getsize(os.path.join(DOCS_PATH, f)) // 1024
    print(f"  {f:<60} {size_kb:>6} KB")

Found 12 PDFs:

  Well Architected - Adaptable - Composable.pdf                  4339 KB
  Well Architected - Adaptable - Introduction.pdf                  42 KB
  Well Architected - Adaptable - Resilient.pdf                   4179 KB
  Well Architected - Easy - Automated.pdf                        4038 KB
  Well Architected - Easy - Engaging.pdf                         4007 KB
  Well Architected - Easy - Intentional.pdf                      4009 KB
  Well Architected - Easy - Introduction.pdf                       45 KB
  Well Architected - Introduction.pdf                            3851 KB
  Well Architected - Trusted - Compliant.pdf                     4058 KB
  Well Architected - Trusted - Introduction.pdf                    46 KB
  Well Architected - Trusted - Reliable.pdf                      4043 KB
  Well Architected - Trusted - Secure.pdf                        4170 KB


In [7]:
# ============================================================
# CELL 4 — Imports
# ============================================================
import re
import pdfplumber
import chromadb
from chromadb.utils import embedding_functions
from openai    import OpenAI
from anthropic import Anthropic
from pydantic  import BaseModel
from langsmith import traceable

print("✅ Imports complete")

✅ Imports complete


In [8]:
# ============================================================
# CELL 5 — PDF extraction function
#
# Why pdfplumber over PyPDF?
# pdfplumber handles multi-column layouts, preserves spacing
# between sections, and extracts tables cleanly.
# PyPDF often smashes columns together into unreadable text.
# ============================================================

def extract_pdf(filepath: str) -> dict:
    """
    Extract clean text from a PDF file.

    What being done here:
    - Read every page
    - Strip headers/footers by ignoring very short lines
    - Preserve paragraph structure with double newlines
    - Track page numbers for metadata (useful for citations later)
    """
    filename = os.path.basename(filepath)
    doc_name = filename.replace(".pdf", "").replace("-", " ").replace("_", " ").title()

    all_text  = []
    page_count = 0

    try:
        with pdfplumber.open(filepath) as pdf:
            page_count = len(pdf.pages)

            for page_num, page in enumerate(pdf.pages):
                text = page.extract_text()
                if not text:
                    continue  # skip blank pages

                # Clean up the extracted text
                lines = text.split("\n")
                clean_lines = []

                for line in lines:
                    line = line.strip()

                    # Skip lines that are likely headers/footers/page numbers
                    # (very short, purely numeric, or repeated boilerplate)
                    if len(line) < 20:
                        continue
                    if line.isdigit():
                        continue

                    clean_lines.append(line)

                if clean_lines:
                    # Add page marker so page numbers can be cited later
                    all_text.append(f"[Page {page_num + 1}]\n" + "\n".join(clean_lines))

        full_text = "\n\n".join(all_text)

        return {
            "filename":   filename,
            "doc_name":   doc_name,
            "filepath":   filepath,
            "text":       full_text,
            "page_count": page_count,
            "char_count": len(full_text),
        }

    except Exception as e:
        print(f"  ❌ Failed to extract {filename}: {e}")
        return None


# Extract all PDFs
print("Extracting text from PDFs...\n")
documents = []

for filename in pdf_files:
    filepath = os.path.join(DOCS_PATH, filename)
    print(f"  Extracting: {filename}")
    doc = extract_pdf(filepath)
    if doc:
        documents.append(doc)
        print(f"    ✅ {doc['doc_name']} — {doc['page_count']} pages, {doc['char_count']:,} chars\n")

print(f"✅ Extracted {len(documents)}/{len(pdf_files)} PDFs successfully")
total_chars = sum(d['char_count'] for d in documents)
print(f"   Total corpus size: {total_chars:,} characters")

Extracting text from PDFs...

  Extracting: Well Architected - Adaptable - Composable.pdf
    ✅ Well Architected   Adaptable   Composable — 30 pages, 44,637 chars

  Extracting: Well Architected - Adaptable - Introduction.pdf
    ✅ Well Architected   Adaptable   Introduction — 2 pages, 2,201 chars

  Extracting: Well Architected - Adaptable - Resilient.pdf
    ✅ Well Architected   Adaptable   Resilient — 44 pages, 79,431 chars

  Extracting: Well Architected - Easy - Automated.pdf
    ✅ Well Architected   Easy   Automated — 21 pages, 31,990 chars

  Extracting: Well Architected - Easy - Engaging.pdf
    ✅ Well Architected   Easy   Engaging — 20 pages, 33,103 chars

  Extracting: Well Architected - Easy - Intentional.pdf
    ✅ Well Architected   Easy   Intentional — 21 pages, 37,446 chars

  Extracting: Well Architected - Easy - Introduction.pdf
    ✅ Well Architected   Easy   Introduction — 2 pages, 2,449 chars

  Extracting: Well Architected - Introduction.pdf
    ✅ Well Architected  

In [9]:
# ============================================================
# CELL 6 — Sanity check: preview one document
# ============================================================

sample = documents[0]

print(f"Document : {sample['doc_name']}")
print(f"Pages    : {sample['page_count']}")
print(f"Chars    : {sample['char_count']:,}")
print(f"\n--- First 1000 characters ---\n")
print(sample["text"][:1000])

Document : Well Architected   Adaptable   Composable
Pages    : 30
Chars    : 44,637

--- First 1000 characters ---

[Page 1]
Composable solutions adjust quickly and with greater stability. A system architected to be
composable is built in meaningful units, or building blocks, that can operate gracefully with
one another and can be easily swapped in and out of service. Building composability into a
system enables you to introduce new features or remove technical debt by refactoring or
reassembling individual units of a system. Composability enables faster, more predictable
delivery cycles and releases, as teams can focus on building and delivering meaningful
features via smaller amounts of change.
Composable systems make it possible for businesses to adapt more quickly and with
greater stability — whether the stimulus is internal to the business or caused by external
factors. Composability helps systems be more resilient and can help make solutions and
architectural patterns more inten

In [10]:
# ============================================================
# CELL 7 — Chunk the text
# ============================================================

def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 100) -> list[str]:
    chunks = []
    start  = 0
    text_len = len(text)

    while start < text_len:
        end = min(start + chunk_size, text_len)  # never go past end of text

        chunk = text[start:end].strip()

        if len(chunk) > 100:
            chunks.append(chunk)

        # Always advance by at least (chunk_size - overlap)
        # This guarantees the loop always moves forward
        next_start = start + chunk_size - overlap

        # Safety guard — if we didn't advance, force it forward
        if next_start <= start:
            next_start = start + 1

        start = next_start

    return chunks


print("Chunking documents...\n")
all_chunks = []

for doc in documents:
    chunks = chunk_text(doc["text"])

    for i, chunk in enumerate(chunks):
        all_chunks.append({
            "id":           f"{doc['filename'].replace('.pdf','').replace(' ','_')}_{i}",
            "text":         chunk,
            "filename":     doc["filename"],
            "doc_name":     doc["doc_name"],
            "chunk_num":    i,
            "total_chunks": len(chunks),
        })

    print(f"  {doc['doc_name'][:55]:<55} → {len(chunks):>3} chunks")

del documents

print(f"\n✅ Total chunks: {len(all_chunks)}")
print(f"   Avg chunk size: {sum(len(c['text']) for c in all_chunks) // len(all_chunks)} chars")

Chunking documents...

  Well Architected   Adaptable   Composable               →  41 chunks
  Well Architected   Adaptable   Introduction             →   2 chunks
  Well Architected   Adaptable   Resilient                →  73 chunks
  Well Architected   Easy   Automated                     →  29 chunks
  Well Architected   Easy   Engaging                      →  31 chunks
  Well Architected   Easy   Intentional                   →  34 chunks
  Well Architected   Easy   Introduction                  →   3 chunks
  Well Architected   Introduction                         →   4 chunks
  Well Architected   Trusted   Compliant                  →  35 chunks
  Well Architected   Trusted   Introduction               →   3 chunks
  Well Architected   Trusted   Reliable                   →  37 chunks
  Well Architected   Trusted   Secure                     →  46 chunks

✅ Total chunks: 338
   Avg chunk size: 1178 chars


In [11]:
# ============================================================
# CELL 8 — Embed chunks and store in Chroma
#
# What happens here:
# 1. Chroma calls OpenAI's embedding API on each chunk
# 2. Each chunk → 1536-dimension vector
# 3. Vectors stored in an in-memory HNSW index
# 4. At query time, question is embedded the same way and
#    Chroma finds closest vectors by cosine similarity
#
# Cost:  ~$0.01-0.03 for the full 27-PDF corpus
# Time:  ~45-90 seconds depending on total chunk count
#
# In-memory = resets on Colab restart (intentional).
# README note: "Production would persist to Pinecone or
# a disk-backed Chroma instance"
# ============================================================

openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name="text-embedding-3-small"
)

chroma_client = chromadb.Client()

try:
    chroma_client.delete_collection("salesforce_docs")
    print("Existing collection cleared")
except:
    pass

collection = chroma_client.create_collection(
    name="salesforce_docs",
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"}
)

BATCH_SIZE    = 50
total_batches = (len(all_chunks) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"\nEmbedding {len(all_chunks)} chunks in {total_batches} batches...\n")

for i in range(0, len(all_chunks), BATCH_SIZE):
    batch     = all_chunks[i:i + BATCH_SIZE]
    batch_num = (i // BATCH_SIZE) + 1

    collection.add(
        ids       = [c["id"]      for c in batch],
        documents = [c["text"]    for c in batch],
        metadatas = [{
            "filename":     c["filename"],
            "doc_name":     c["doc_name"],
            "chunk_num":    c["chunk_num"],
            "total_chunks": c["total_chunks"],
        } for c in batch],
    )

    print(f"  Batch {batch_num:>2}/{total_batches} ✅  ({len(batch)} chunks)")

print(f"\n✅ Embedding complete")
print(f"   Collection size: {collection.count()} chunks")


Embedding 338 chunks in 7 batches...

  Batch  1/7 ✅  (50 chunks)
  Batch  2/7 ✅  (50 chunks)
  Batch  3/7 ✅  (50 chunks)
  Batch  4/7 ✅  (50 chunks)
  Batch  5/7 ✅  (50 chunks)
  Batch  6/7 ✅  (50 chunks)
  Batch  7/7 ✅  (38 chunks)

✅ Embedding complete
   Collection size: 338 chunks


In [12]:
# ============================================================
# CELL 9 — Pydantic models
# ============================================================

class RAGQuery(BaseModel):
    question: str
    top_k:    int = 5
    provider: str = "openai"


class RetrievedChunk(BaseModel):
    text:      str
    doc_name:  str
    filename:  str
    chunk_num: int
    distance:  float


class RAGResponse(BaseModel):
    question:    str
    answer:      str
    provider:    str
    model:       str
    chunks_used: list[RetrievedChunk]
    chunk_count: int


print("✅ Pydantic models defined")

✅ Pydantic models defined


In [13]:
# ============================================================
# CELL 10 — Retrieval function
#
# Chroma embeds the question using the same OpenAI model
# used at ingest time — critical, must be the same model.
# HNSW index finds top-k closest vectors by cosine distance.
#
# Distance interpretation:
#   < 0.3  → strong match
#   0.3-0.5 → okay match
#   > 0.6  → weak — question may be out of scope
# ============================================================

@traceable(name="Retrieve Chunks")
def retrieve(question: str, top_k: int = 5) -> list[RetrievedChunk]:
    results = collection.query(
        query_texts=[question],
        n_results=top_k,
        include=["documents", "metadatas", "distances"]
    )

    chunks = []
    for text, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunks.append(RetrievedChunk(
            text=text,
            doc_name=meta["doc_name"],
            filename=meta["filename"],
            chunk_num=meta["chunk_num"],
            distance=round(dist, 4)
        ))

    return chunks


print("✅ Retrieval function defined")

✅ Retrieval function defined


In [14]:
# ============================================================
# CELL 11 — System prompt and context builder
# ============================================================

SYSTEM_PROMPT = """You are a senior Salesforce Solution Architect with deep expertise
in platform architecture, data modeling, integrations, security, and Agentforce.

Answer questions using ONLY the context chunks provided below from official
Salesforce architecture documentation.

Rules:
- Use ONLY the provided context — not your general training knowledge
- Cite sources using [Source: document name] inline in your answer
- If the context lacks sufficient information, say so explicitly
- Be specific and technical — the user is an experienced Salesforce architect
- No filler sentences — be direct and complete"""


def build_context(chunks: list[RetrievedChunk]) -> str:
    parts = []
    for i, chunk in enumerate(chunks):
        parts.append(f"[Source {i+1}: {chunk.doc_name}]\n{chunk.text}")
    return "\n\n---\n\n".join(parts)


print("✅ System prompt and context builder defined")

✅ System prompt and context builder defined


In [15]:
# ============================================================
# CELL 12 — Generation functions
#
# temperature=0.2: low randomness = factual, consistent answers
# Correct for a documentation Q&A system — same question
# should return the same answer every time.
# ============================================================

@traceable(name="Generate — OpenAI")
def generate_openai(question: str, chunks: list[RetrievedChunk]) -> tuple[str, str]:
    client  = OpenAI()
    context = build_context(chunks)

    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        max_tokens=600,
        temperature=0.2
    )
    return resp.choices[0].message.content, "gpt-4o-mini"


@traceable(name="Generate — Anthropic")
def generate_anthropic(question: str, chunks: list[RetrievedChunk]) -> tuple[str, str]:
    client  = Anthropic()
    context = build_context(chunks)

    resp = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        system=SYSTEM_PROMPT,
        messages=[
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ]
    )
    return resp.content[0].text, "claude-haiku-4-5-20251001"


print("✅ Generation functions defined")

✅ Generation functions defined


In [16]:
# ============================================================
# CELL 13 — Full RAG pipeline
#
# Orchestrates retrieval + generation into one clean call.
# @ traceable on rag() creates a parent trace in LangSmith.
# Child traces from retrieve() and generate_*() nest under it.
# You see the full chain: question → chunks → answer.
# ============================================================

@traceable(name="RAG Pipeline")
def rag(query: RAGQuery) -> RAGResponse:
    chunks = retrieve(question=query.question, top_k=query.top_k)

    if query.provider == "anthropic":
        answer, model = generate_anthropic(query.question, chunks)
    else:
        answer, model = generate_openai(query.question, chunks)

    return RAGResponse(
        question=query.question,
        answer=answer,
        provider=query.provider,
        model=model,
        chunks_used=chunks,
        chunk_count=len(chunks)
    )


def ask(question: str, provider: str = "openai", top_k: int = 5) -> RAGResponse:
    response = rag(RAGQuery(question=question, provider=provider, top_k=top_k))

    print(f"\n{'='*62}")
    print(f"  Q        : {response.question}")
    print(f"  Provider : {response.provider} ({response.model})")
    print(f"  Chunks   : {response.chunk_count} retrieved")
    print(f"\n  Answer:\n")
    for line in response.answer.strip().split("\n"):
        print(f"    {line}")
    print(f"\n  Sources (lower distance = stronger match):")
    for chunk in response.chunks_used:
        quality = "✅" if chunk.distance < 0.3 else "⚠️" if chunk.distance > 0.5 else "🟡"
        print(f"    {quality} [{chunk.distance:.3f}] {chunk.doc_name}  (chunk {chunk.chunk_num})")
    print(f"{'='*62}\n")

    return response


print("✅ RAG pipeline ready")

✅ RAG pipeline ready


In [17]:
# ============================================================
# CELL 14 — Test with 5 Salesforce architecture questions
# ============================================================
import time

TEST_QUESTIONS = [
    "What are the core principles of the Salesforce Well Architected Framework?",
    "What does it mean for a Salesforce solution to be trusted?",
    "How should I approach building composable architecture on Salesforce?",
    "What are the key principles for building reliable Salesforce applications?",
    "How does the Well Architected Framework define a secure Salesforce solution?",
]

print("Running 5 test questions...\n")

for question in TEST_QUESTIONS:
    ask(question, provider="openai", top_k=5)
    time.sleep(1)

Running 5 test questions...


  Q        : What are the core principles of the Salesforce Well Architected Framework?
  Provider : openai (gpt-4o-mini)
  Chunks   : 5 retrieved

  Answer:

    The core principles of the Salesforce Well Architected Framework are:
    
    1. **Trusted**: Solutions protect business and stakeholders through security, compliance, and reliability.
    2. **Easy**: Solutions deliver business value quickly by being intentional, automated, and engaging.
    3. **Adaptable**: Solutions evolve with the business, allowing for flexibility and responsiveness to changing needs.
    
    These principles guide architects in building healthy solutions on the Salesforce Customer 360 Platform [Source: Well Architected Introduction].

  Sources (lower distance = stronger match):
    ✅ [0.223] Well Architected   Introduction  (chunk 0)
    ✅ [0.260] Well Architected   Introduction  (chunk 1)
    🟡 [0.357] Well Architected   Introduction  (chunk 2)
    🟡 [0.387] Well Archi

In [18]:
# ============================================================
# CELL 15 — Side-by-side provider comparison
#
# Same question, same retrieved chunks, different LLMs. This RAG
# pipeline is provider-agnostic
# ============================================================
import time

COMPARISON_QUESTION = "What are the core architectural principles of the Salesforce Platform and how do they support enterprise-grade applications?"

print("OpenAI vs Anthropic — same retrieval, different generation\n")

r_openai    = ask(COMPARISON_QUESTION, provider="openai")
time.sleep(1)
r_anthropic = ask(COMPARISON_QUESTION, provider="anthropic")

openai_sources    = {c.doc_name for c in r_openai.chunks_used}
anthropic_sources = {c.doc_name for c in r_anthropic.chunks_used}

print("\n📊 Source overlap:")
print(f"  Shared           : {openai_sources & anthropic_sources}")
print(f"  Unique to OpenAI : {openai_sources - anthropic_sources}")
print(f"  Unique to Claude : {anthropic_sources - openai_sources}")

OpenAI vs Anthropic — same retrieval, different generation


  Q        : What are the core architectural principles of the Salesforce Platform and how do they support enterprise-grade applications?
  Provider : openai (gpt-4o-mini)
  Chunks   : 5 retrieved

  Answer:

    The core architectural principles of the Salesforce Platform, as outlined in the Salesforce Well-Architected Framework, are Trusted, Easy, and Adaptable. These principles support enterprise-grade applications in the following ways:
    
    1. **Trusted**: This principle emphasizes the importance of protecting stakeholders and data. Secure architectures verify user identities, restrict data access, and prevent data breaches. By prioritizing customer trust and data security, Salesforce ensures that organizations can build secure solutions that safeguard their data and comply with regulatory requirements [Source: Well Architected Trusted Secure].
    
    2. **Easy**: Solutions built on the Salesforce Customer 360 Plat

In [19]:
# ============================================================
# CELL 16 — Debug retrieval
#
# Run this when answers feel off or incomplete.
# Shows raw chunks that are retrieved
# and whether distance scores make sense.
# ============================================================

def debug_retrieval(question: str, top_k: int = 5):
    chunks = retrieve(question, top_k=top_k)

    print(f"Question : {question}")
    print(f"Retrieved: {len(chunks)} chunks\n")

    for i, chunk in enumerate(chunks):
        quality = "✅ strong" if chunk.distance < 0.3 else "⚠️  weak" if chunk.distance > 0.5 else "🟡 okay"
        print(f"--- Chunk {i+1} [{quality}] ---")
        print(f"Source   : {chunk.doc_name}")
        print(f"File     : {chunk.filename}")
        print(f"Distance : {chunk.distance}")
        print(f"Preview  : {chunk.text[:300]}...\n")


debug_retrieval("What governor limits apply to async Apex?")

Question : What governor limits apply to async Apex?
Retrieved: 5 chunks

--- Chunk 1 [⚠️  weak] ---
Source   : Well Architected   Trusted   Reliable
File     : Well Architected - Trusted - Reliable.pdf
Distance : 0.533
Preview  : number of users who must access your system simultaneously
• The overall complexity of transaction logic in the system
When thinking about performance, teams sometimes focus too narrowly on compute and
the constraints on maximum CPU time, which are among the platform’s governor limits.
Teams with a ...

--- Chunk 2 [⚠️  weak] ---
Source   : Well Architected   Easy   Automated
File     : Well Architected - Easy - Automated.pdf
Distance : 0.5352
Preview  : oritization
logic for all data sources and data logic for data sources and data
lake objects exists lake objects are not included
- Field mapping from data lake object - Field mapping from data lake
to data model object exists objects to data model objects is
In your Apex: In your Apex:
- All synchr...

--- C

In [20]:
# ============================================================
# CELL 17 — Interactive: ask your own question
# ============================================================

MY_QUESTION = "How Topics Are Arranged"

result = ask(MY_QUESTION, provider="openai", top_k=5)

# Uncomment to compare both providers:
# result_claude = ask(MY_QUESTION, provider="anthropic", top_k=5)


  Q        : How Topics Are Arranged
  Provider : openai (gpt-4o-mini)
  Chunks   : 5 retrieved

  Answer:

    Topics in the Salesforce Well-Architected framework are arranged in a two-level structure. The top level categorizes well-architected solutions into three main attributes: Trusted, Easy, and Adaptable. The second level provides detailed considerations, patterns to build, anti-patterns to avoid, and prescriptive guidance related to specific aspects of these attributes. This organization aims to drive higher quality and longer-lasting health in Salesforce Customer 360 applications. 
    
    - **Trusted**: Focuses on protecting the business and stakeholders through security, compliance, and reliability.
    - **Easy**: Aims to deliver business value quickly through intentionality, automation, and engagement.
    - **Adaptable**: Emphasizes composability and flexibility in design and architecture [Source: Well Architected Introduction].

  Sources (lower distance = stronger mat

In [21]:
# ============================================================
# CELL 18 — Install DeepEval
# ============================================================
!pip install deepeval --quiet

print("✅ DeepEval installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 965.1/965.1 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.5/110.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.0/264.0 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
✅ DeepEval installed


In [22]:
# ============================================================
# CELL 19 — Imports for DeepEval
# ============================================================
from deepeval import evaluate
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    ContextualRecallMetric,
)
from deepeval.test_case import LLMTestCase

print("✅ DeepEval imports complete")

✅ DeepEval imports complete


In [23]:
# ============================================================
# CELL 20 — Configure DeepEval to use OpenAI
# ============================================================
import os
from deepeval.models import GPTModel

# DeepEval uses OpenAI under the hood for evaluation
# It picks up OPENAI_API_KEY from environment automatically
os.environ["OPENAI_API_KEY"] = os.environ["OPENAI_API_KEY"]  # already set

print("✅ DeepEval configured")

✅ DeepEval configured


In [24]:
# ============================================================
# CELL 21 — Golden test set (20 questions)
# ============================================================

golden_test_set = [
    {
        "question": "What are the three pillars of the Salesforce Well-Architected Framework?",
        "ground_truth": "The three pillars are Trusted, Easy, and Adaptable."
    },
    {
        "question": "What does it mean for a Salesforce solution to be trusted?",
        "ground_truth": "A trusted solution is secure, compliant, and reliable. It protects business and stakeholders by controlling access, following legal and ethical guidelines, and operating efficiently and dependably."
    },
    {
        "question": "What are the three qualities of an Easy Salesforce solution?",
        "ground_truth": "An easy solution is intentional, automated, and engaging. It delivers business value immediately, enables work to get done faster at scale, and delights users to drive adoption."
    },
    {
        "question": "What does it mean for a Salesforce solution to be adaptable?",
        "ground_truth": "An adaptable solution is resilient and composable. It handles change well, adjusts quickly with greater stability, and helps businesses innovate and thrive in times of disruption."
    },
    {
        "question": "How does the Well-Architected Framework define an intentional solution?",
        "ground_truth": "An intentional solution delivers business value immediately and over time by making deliberate architectural decisions that align with business goals."
    },
    {
        "question": "What is the purpose of the Salesforce Well-Architected Framework?",
        "ground_truth": "The Well-Architected Framework helps architects build Salesforce solutions that deliver value fast, create and maintain confidence, and evolve with the business."
    },
    {
        "question": "What does compliance mean in the context of the Well-Architected Framework?",
        "ground_truth": "Compliance means following legal and ethical guidelines, protecting business and stakeholders by adhering to regulatory requirements and organizational policies."
    },
    {
        "question": "What makes a Salesforce solution reliable?",
        "ground_truth": "A reliable solution operates efficiently and dependably, maintaining consistent performance and availability to support business operations."
    },
    {
        "question": "What does security mean in the Well-Architected Framework?",
        "ground_truth": "Security means controlling access to protect business and stakeholders, ensuring that only authorized users can access appropriate data and functionality."
    },
    {
        "question": "How does the Well-Architected Framework define an automated solution?",
        "ground_truth": "An automated solution enables work to get done faster at scale by reducing manual effort and streamlining processes through automation."
    },
    {
        "question": "What does it mean for a Salesforce solution to be engaging?",
        "ground_truth": "An engaging solution delights users to drive adoption, creating experiences that are intuitive and valuable to the people using them."
    },
    {
        "question": "What does composable mean in the context of Salesforce architecture?",
        "ground_truth": "A composable solution is built from reusable, modular components that can be assembled and reassembled to meet changing business needs."
    },
    {
        "question": "What does resilient mean in the context of Salesforce architecture?",
        "ground_truth": "A resilient solution handles change well and adjusts quickly with greater stability, maintaining functionality and performance under changing conditions."
    },
    {
        "question": "How do the three pillars of the Well-Architected Framework relate to each other?",
        "ground_truth": "The three pillars — Trusted, Easy, and Adaptable — work together as complementary qualities. A well-architected solution balances all three to deliver value, maintain confidence, and evolve with the business."
    },
    {
        "question": "What role does the Well-Architected Framework play in architectural decision making?",
        "ground_truth": "The framework serves as a set of compass points that architects can return to when evaluating complex architectural details and trade-offs, helping guide design decisions."
    },
    {
        "question": "What is the relationship between trust and security in the Well-Architected Framework?",
        "ground_truth": "Security is one of the three components of trust alongside compliance and reliability. Together they form the Trusted pillar which creates and maintains confidence in a Salesforce solution."
    },
    {
        "question": "How does the Well-Architected Framework support business innovation?",
        "ground_truth": "Through the Adaptable pillar, the framework helps businesses build solutions that are resilient and composable, enabling them to innovate and thrive in times of disruption."
    },
    {
        "question": "What is the significance of user adoption in the Well-Architected Framework?",
        "ground_truth": "User adoption is addressed through the Easy pillar, specifically the engaging quality which focuses on delighting users to drive adoption of Salesforce solutions."
    },
    {
        "question": "How does the Well-Architected Framework address scalability?",
        "ground_truth": "Scalability is addressed through the Easy pillar's automated quality, which enables work to get done faster at scale by reducing manual effort and streamlining processes."
    },
    {
        "question": "What types of guidance does the Well-Architected Framework provide?",
        "ground_truth": "The framework provides prescriptive guidance, patterns to build, anti-patterns to avoid, and considerations related to specific aspects of Trusted, Easy, and Adaptable solutions."
    },
]

print(f"✅ Golden test set ready — {len(golden_test_set)} questions")

✅ Golden test set ready — 20 questions


In [25]:
# ============================================================
# CELL 22 — Run RAG pipeline on all 20 questions
#
# Collects question, answer, retrieved context, and
# ground truth for every item in the golden test set
# ============================================================
import time

print("Running RAG pipeline on 20 questions...\n")

test_cases    = []
raw_results   = []

for i, item in enumerate(golden_test_set):
    question = item["question"]
    print(f"  [{i+1}/20] {question[:65]}...")

    # Retrieve chunks
    chunks = retrieve(question, top_k=5)

    # Generate answer
    query    = RAGQuery(question=question, provider="openai", top_k=5)
    response = rag(query)

    # Store raw results for comparison later
    raw_results.append({
        "question":     question,
        "answer":       response.answer,
        "contexts":     [c.text for c in response.chunks_used],
        "ground_truth": item["ground_truth"],
    })

    # Build DeepEval test case
    test_case = LLMTestCase(
        input=question,
        actual_output=response.answer,
        retrieval_context=[c.text for c in response.chunks_used],
        expected_output=item["ground_truth"],
    )
    test_cases.append(test_case)

    time.sleep(0.5)

print(f"\n✅ Pipeline run complete — {len(test_cases)} test cases built")

Running RAG pipeline on 20 questions...

  [1/20] What are the three pillars of the Salesforce Well-Architected Fra...
  [2/20] What does it mean for a Salesforce solution to be trusted?...
  [3/20] What are the three qualities of an Easy Salesforce solution?...
  [4/20] What does it mean for a Salesforce solution to be adaptable?...
  [5/20] How does the Well-Architected Framework define an intentional sol...
  [6/20] What is the purpose of the Salesforce Well-Architected Framework?...
  [7/20] What does compliance mean in the context of the Well-Architected ...
  [8/20] What makes a Salesforce solution reliable?...
  [9/20] What does security mean in the Well-Architected Framework?...
  [10/20] How does the Well-Architected Framework define an automated solut...
  [11/20] What does it mean for a Salesforce solution to be engaging?...
  [12/20] What does composable mean in the context of Salesforce architectu...
  [13/20] What does resilient mean in the context of Salesforce architect

In [26]:
# ============================================================
# CELL 23 — Run DeepEval evaluation (raw pipeline)
#
# Faithfulness    — is answer grounded in retrieved context?
# AnswerRelevancy — does answer address the question?
# ContextualRecall — did retriever find the right chunks?
#
# Each metric uses GPT-4o-mini under the hood to judge quality
# This takes 3-5 minutes to run across 20 questions
# ============================================================

print("Initialising metrics...")

faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

recall_metric = ContextualRecallMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

print("Running evaluation across 20 test cases...")
print("This takes 3-5 minutes — each question is judged by GPT-4o-mini\n")

results = evaluate(
    test_cases=test_cases,
    metrics=[faithfulness_metric, relevancy_metric, recall_metric],
)

print("\n✅ Evaluation complete")

Initialising metrics...
Running evaluation across 20 test cases...
This takes 3-5 minutes — each question is judged by GPT-4o-mini



✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 3 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=630913;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 21.94s | token cost: 0.0340911 USD)
» Test Results (20 total tests):
   » Pass Rate: 95.0% | Passed: 19 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


✅ Evaluation complete


In [27]:
# ============================================================
# CELL 24 — Print scores (raw pipeline baseline)
# ============================================================

# DeepEval stores results in results.test_results
faith_scores  = []
relev_scores  = []
recall_scores = []

for test_result in results.test_results:
    for metric_data in test_result.metrics_data:
        name  = metric_data.name
        score = metric_data.score
        if score is None:
            continue
        if "Faithfulness" in name:
            faith_scores.append(score)
        elif "Answer Relevancy" in name or "AnswerRelevancy" in name:
            relev_scores.append(score)
        elif "Contextual Recall" in name or "ContextualRecall" in name:
            recall_scores.append(score)

avg_faith  = sum(faith_scores)  / len(faith_scores)  if faith_scores  else 0
avg_relev  = sum(relev_scores)  / len(relev_scores)  if relev_scores  else 0
avg_recall = sum(recall_scores) / len(recall_scores) if recall_scores else 0

print("=" * 50)
print("RAW PIPELINE — DEEPEVAL SCORES")
print("=" * 50)
print(f"  Faithfulness:     {avg_faith:.4f}")
print(f"  Answer Relevancy: {avg_relev:.4f}")
print(f"  Context Recall:   {avg_recall:.4f}")
print("=" * 50)

raw_scores = {
    "faithfulness":     avg_faith,
    "answer_relevancy": avg_relev,
    "context_recall":   avg_recall,
}

print("\nScores saved to raw_scores for LangChain comparison")

RAW PIPELINE — DEEPEVAL SCORES
  Faithfulness:     0.9315
  Answer Relevancy: 0.9774
  Context Recall:   1.0000

Scores saved to raw_scores for LangChain comparison


In [28]:
# ============================================================
# CELL 25 — Per-question breakdown
# ============================================================

print(f"\n{'Question':<58} {'Faith':>6} {'Relev':>6} {'Recall':>7}")
print("-" * 82)

for test_result in results.test_results:
    q = test_result.input[:55] + "..."

    faith  = "N/A"
    relev  = "N/A"
    recall = "N/A"

    for metric_data in test_result.metrics_data:
        name  = metric_data.name
        score = metric_data.score
        if score is None:
            continue
        if "Faithfulness" in name:
            faith = f"{score:.2f}"
        elif "Answer Relevancy" in name or "AnswerRelevancy" in name:
            relev = f"{score:.2f}"
        elif "Contextual Recall" in name or "ContextualRecall" in name:
            recall = f"{score:.2f}"

    print(f"  {q:<58} {faith:>6} {relev:>6} {recall:>7}")


Question                                                    Faith  Relev  Recall
----------------------------------------------------------------------------------
  What are the three pillars of the Salesforce Well-Archi...   1.00   1.00    1.00
  What is the relationship between trust and security in ...   1.00   1.00    1.00
  What does composable mean in the context of Salesforce ...   1.00   1.00    1.00
  How do the three pillars of the Well-Architected Framew...   1.00   1.00    1.00
  How does the Well-Architected Framework support busines...   1.00   1.00    1.00
  What does it mean for a Salesforce solution to be adapt...   1.00   1.00    1.00
  How does the Well-Architected Framework define an autom...   1.00   0.86    1.00
  What are the three qualities of an Easy Salesforce solu...   1.00   1.00    1.00
  What does it mean for a Salesforce solution to be trust...   1.00   1.00    1.00
  What makes a Salesforce solution reliable?...                1.00   1.00    1.00
  Wha

In [29]:
# ============================================================
# CELL 26 — Install LangChain dependencies
# ============================================================
!pip install langchain langchain-openai langchain-community langchain-text-splitters pypdf --quiet

print("✅ LangChain dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
opentelemetry-exporter-gcp-logging 1.11.0a0 requires opentelemetry-sdk<1.39.0,>=1.35.0, but you have opentelemetry-sdk 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.42.1 which is incompatible.
google-adk 1.29.0 requires opentelemetry-sdk<1.39.0,>=1.36.0, bu

In [30]:
# ============================================================
# CELL 27 — LangChain imports (current API, no deprecated modules)
# ============================================================
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma as LangChainChroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

print("✅ LangChain imports complete")

/tmp/ipykernel_3703/1019622255.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ LangChain imports complete


In [31]:
# ============================================================
# CELL 28 — Load and split PDFs using LangChain
#
# What's different from your raw pipeline:
#
# RAW:      pdfplumber opens each PDF page by page,
#           you manually clean lines, join pages,
#           then chunk with your own chunk_text() function
#
# LANGCHAIN: PyPDFLoader handles extraction automatically,
#            RecursiveCharacterTextSplitter handles chunking
#            It tries to split on paragraphs first (\n\n),
#            then sentences (\n), then words ( )
#            before falling back to character splits
#            This is smarter boundary detection than your
#            fixed-size approach — but you now have less
#            control over exactly what happens
# ============================================================
import os
from pathlib import Path

DOCS_PATH = f"/content/drive/MyDrive/{os.environ.get('GOOGLE_DRIVE_FOLDER', 'salesforce-rag-docs')}"

print("Loading PDFs with LangChain...\n")

pdf_files = sorted([f for f in os.listdir(DOCS_PATH) if f.endswith(".pdf")])
lc_documents = []

for filename in pdf_files:
    filepath = os.path.join(DOCS_PATH, filename)
    loader   = PyPDFLoader(filepath)
    pages    = loader.load()
    lc_documents.extend(pages)
    print(f"  ✅ {filename} — {len(pages)} pages")

print(f"\n✅ Loaded {len(lc_documents)} total pages from {len(pdf_files)} PDFs")

Loading PDFs with LangChain...

  ✅ Well Architected - Adaptable - Composable.pdf — 30 pages
  ✅ Well Architected - Adaptable - Introduction.pdf — 2 pages
  ✅ Well Architected - Adaptable - Resilient.pdf — 44 pages
  ✅ Well Architected - Easy - Automated.pdf — 21 pages
  ✅ Well Architected - Easy - Engaging.pdf — 20 pages
  ✅ Well Architected - Easy - Intentional.pdf — 21 pages
  ✅ Well Architected - Easy - Introduction.pdf — 2 pages
  ✅ Well Architected - Introduction.pdf — 3 pages
  ✅ Well Architected - Trusted - Compliant.pdf — 25 pages
  ✅ Well Architected - Trusted - Introduction.pdf — 2 pages
  ✅ Well Architected - Trusted - Reliable.pdf — 24 pages
  ✅ Well Architected - Trusted - Secure.pdf — 34 pages

✅ Loaded 228 total pages from 12 PDFs


In [32]:
# ============================================================
# CELL 29 — Split into chunks using RecursiveCharacterTextSplitter
#
# Using same chunk_size (1200) and overlap (100) as
# raw pipeline so the comparison is fair — only the
# splitting strategy changes, not the size parameters
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
    # Tries each separator in order
    # \n\n = paragraph break (preferred)
    # \n   = line break
    # .    = sentence end
    # " "  = word boundary
    # ""   = character (last resort)
)

lc_chunks = splitter.split_documents(lc_documents)

print(f"✅ LangChain chunking complete")
print(f"   Total chunks: {len(lc_chunks)}")
print(f"   Avg chunk size: {sum(len(c.page_content) for c in lc_chunks) // len(lc_chunks)} chars")
print(f"\n   Compare to raw pipeline: 338 chunks")
print(f"   Difference: {len(lc_chunks) - 338:+d} chunks")

✅ LangChain chunking complete
   Total chunks: 458
   Avg chunk size: 881 chars

   Compare to raw pipeline: 338 chunks
   Difference: +120 chunks


In [33]:
# ============================================================
# CELL 30 — Embed and store using LangChain + Chroma
#
# Same embedding model (text-embedding-3-small)
# Same vector DB (Chroma)
# Same similarity metric (cosine)
# Only difference: LangChain manages the embedding calls
# instead of chromadb.utils.embedding_functions
# ============================================================

print("Building LangChain vector store...")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

lc_vectorstore = LangChainChroma.from_documents(
    documents=lc_chunks,
    embedding=embeddings,
    collection_name="salesforce_docs_langchain",
    collection_metadata={"hnsw:space": "cosine"}
)

print(f"✅ LangChain vector store built")
print(f"   Collection size: {lc_vectorstore._collection.count()} chunks")

Building LangChain vector store...
✅ LangChain vector store built
   Collection size: 458 chunks


In [34]:
# ============================================================
# CELL 31 — Build LangChain RAG chain (LCEL approach)
# ============================================================

LANGCHAIN_PROMPT = PromptTemplate(
    template="""You are a senior Salesforce Solution Architect with deep expertise
in the Salesforce Well Architected Framework.

Answer questions using ONLY the context provided from official Salesforce
Well Architected documentation.

Rules:
- Use ONLY the provided context — not your general training knowledge
- Cite sources inline using [Source: document name]
- If the context lacks sufficient information, say so explicitly
- Be specific and technical — the user is an experienced Salesforce architect
- No filler sentences — be direct and complete

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

lc_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

lc_retriever = lc_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Build context and question inputs separately to avoid
# the dict syntax issue inside the pipe operator
context_chain  = lc_retriever | format_docs
question_chain = RunnablePassthrough()

lc_chain = (
    {"context": context_chain, "question": question_chain}
    | LANGCHAIN_PROMPT
    | lc_llm
    | StrOutputParser()
)

print("✅ LangChain LCEL chain ready")

✅ LangChain LCEL chain ready


In [35]:
# ============================================================
# CELL 32 — Run LangChain pipeline on same 20 questions
# ============================================================
import time

print("Running LangChain pipeline on 20 questions...\n")

lc_test_cases = []

for i, item in enumerate(golden_test_set):
    question = item["question"]
    print(f"  [{i+1}/20] {question[:65]}...")

    # Run through LangChain LCEL chain — returns string directly
    answer = lc_chain.invoke(question)

    # Get retrieved docs separately for DeepEval context
    retrieved_docs = lc_retriever.invoke(question)

    # Build DeepEval test case
    test_case = LLMTestCase(
        input=question,
        actual_output=answer,
        retrieval_context=[doc.page_content for doc in retrieved_docs],
        expected_output=item["ground_truth"],
    )
    lc_test_cases.append(test_case)

    time.sleep(0.5)

print(f"\n✅ LangChain pipeline run complete — {len(lc_test_cases)} test cases")

Running LangChain pipeline on 20 questions...

  [1/20] What are the three pillars of the Salesforce Well-Architected Fra...
  [2/20] What does it mean for a Salesforce solution to be trusted?...
  [3/20] What are the three qualities of an Easy Salesforce solution?...
  [4/20] What does it mean for a Salesforce solution to be adaptable?...
  [5/20] How does the Well-Architected Framework define an intentional sol...
  [6/20] What is the purpose of the Salesforce Well-Architected Framework?...
  [7/20] What does compliance mean in the context of the Well-Architected ...
  [8/20] What makes a Salesforce solution reliable?...
  [9/20] What does security mean in the Well-Architected Framework?...
  [10/20] How does the Well-Architected Framework define an automated solut...
  [11/20] What does it mean for a Salesforce solution to be engaging?...
  [12/20] What does composable mean in the context of Salesforce architectu...
  [13/20] What does resilient mean in the context of Salesforce arc

In [36]:
# ============================================================
# CELL 33 — Evaluate LangChain pipeline with DeepEval
# ============================================================

print("Running DeepEval on LangChain pipeline...\n")

lc_faithfulness_metric = FaithfulnessMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

lc_relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

lc_recall_metric = ContextualRecallMetric(
    threshold=0.7,
    model="gpt-4o-mini",
    include_reason=True
)

lc_results = evaluate(
    test_cases=lc_test_cases,
    metrics=[lc_faithfulness_metric, lc_relevancy_metric, lc_recall_metric],
)

print("\n✅ LangChain evaluation complete")

Running DeepEval on LangChain pipeline...



✨ You're running DeepEval's latest Faithfulness Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a_execute_llm_test_cases
INFO:deepeval.evaluate.execute.e2e:in _a

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_1 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_2 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_3 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_4 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_5 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_6 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_7 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_8 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_9 (Passed 3 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_10 (Passed 3 metrics)                                

⚠ WARNING: No hyperparameters logged.
» ]8;id=358289;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 20.65s | token cost: 0.03177975 USD)
» Test Results (20 total tests):
   » Pass Rate: 100.0% | Passed: 20 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.


✅ LangChain evaluation complete


In [37]:
# ============================================================
# CELL 34 — LangChain scores
# ============================================================

lc_faith_scores  = []
lc_relev_scores  = []
lc_recall_scores = []

for test_result in lc_results.test_results:
    for metric_data in test_result.metrics_data:
        name  = metric_data.name
        score = metric_data.score
        if score is None:
            continue
        if "Faithfulness" in name:
            lc_faith_scores.append(score)
        elif "Answer Relevancy" in name or "AnswerRelevancy" in name:
            lc_relev_scores.append(score)
        elif "Contextual Recall" in name or "ContextualRecall" in name:
            lc_recall_scores.append(score)

lc_avg_faith  = sum(lc_faith_scores)  / len(lc_faith_scores)  if lc_faith_scores  else 0
lc_avg_relev  = sum(lc_relev_scores)  / len(lc_relev_scores)  if lc_relev_scores  else 0
lc_avg_recall = sum(lc_recall_scores) / len(lc_recall_scores) if lc_recall_scores else 0

lc_scores = {
    "faithfulness":     lc_avg_faith,
    "answer_relevancy": lc_avg_relev,
    "context_recall":   lc_avg_recall,
}

print("=" * 60)
print("SIDE-BY-SIDE COMPARISON — RAW vs LANGCHAIN")
print("=" * 60)
print(f"{'Metric':<20} {'Raw Pipeline':>14} {'LangChain':>12} {'Delta':>8}")
print("-" * 60)

for metric in ["faithfulness", "answer_relevancy", "context_recall"]:
    raw = raw_scores[metric]
    lc  = lc_scores[metric]
    delta = lc - raw
    arrow = "▲" if delta > 0 else "▼" if delta < 0 else "="
    print(f"  {metric:<18} {raw:>14.4f} {lc:>12.4f} {arrow}{abs(delta):>6.4f}")

print("=" * 60)
print("\nSave these numbers — they go in your README.")

SIDE-BY-SIDE COMPARISON — RAW vs LANGCHAIN
Metric                 Raw Pipeline    LangChain    Delta
------------------------------------------------------------
  faithfulness               0.9315       0.9522 ▲0.0207
  answer_relevancy           0.9774       0.9938 ▲0.0164
  context_recall             1.0000       1.0000 =0.0000

Save these numbers — they go in your README.
